In [1]:
import pandas as pd
import numpy as np

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

/mnt/d/coding/kaggle/house-price


In [104]:
import importlib

import src.data_loader
importlib.reload(src.data_loader)

from src.data_loader import load_data

In [146]:
train_raw = load_data('../data/raw/train.csv')
test_raw = load_data('../data/raw/test.csv')

In [147]:
print(train_raw.shape)
print(test_raw.shape)

(1460, 81)
(1459, 80)


In [148]:
y_train = np.log1p(train_raw["SalePrice"])
train_raw = train_raw.drop(columns=["Id","SalePrice"])
numeric_featureList = train_raw.select_dtypes(include='number')
x_train = numeric_featureList
print(x_train.shape, y_train.shape)

test_features = test_raw.drop(columns=["Id"])
x_test = test_features.select_dtypes(include='number')
print(x_test.shape)

(1460, 36) (1460,)
(1459, 36)


In [132]:
def check_empty_perc(data):
    for cols in data.columns:
        if data[cols].isnull().sum().astype(int) != 0:
            print((100 * data[cols].isnull().sum()) / data[cols].count(), cols)

In [134]:
# check_empty_perc(x_train)
# check_empty_perc(x_test)

18.425324675324674 LotFrontage
1.0387811634349031 MasVnrArea
0.06858710562414266 BsmtFinSF1
0.06858710562414266 BsmtFinSF2
0.06858710562414266 BsmtUnfSF
0.06858710562414266 TotalBsmtSF
0.13726835964310227 BsmtFullBath
0.13726835964310227 BsmtHalfBath
5.6480811006517015 GarageYrBlt
0.06858710562414266 GarageCars
0.06858710562414266 GarageArea


In [139]:
# x_train_medians = x_train.median()
# print(x_train_medians.shape)

(36,)


In [136]:
# x_train = x_train.fillna(x_train_medians)

In [137]:
# x_test = x_test.fillna(x_train_medians)

In [138]:
# check_empty_perc(x_train)
# check_empty_perc(x_test)

In [180]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [187]:
rmse_err = []

for train_idx, val_idx in kf.split(x_train):
    x_tr = x_train.iloc[train_idx]
    x_val = x_train.iloc[val_idx]
    
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]
    
#     print(x_tr.shape, x_val.shape, y_tr.shape, y_val.shape)
    
#     check_empty_perc(x_tr)
#     check_empty_perc(x_val)
    fold_medians = x_tr.median()
    x_tr = x_tr.fillna(fold_medians)
    x_val = x_val.fillna(fold_medians)
#     check_empty_perc(x_tr)
#     check_empty_perc(x_val)
    
    model = Ridge(random_state=42)
    model.fit(x_tr, y_tr)
    y_pred = model.predict(x_val)
    
    rmse_err.append(np.sqrt(mean_squared_error(y_pred, y_val)))
    print(y_pred.max(), y_pred.min(), y_val.max(), y_val.min())

13.308879388234248 11.010641726930661 13.534474352733596 10.471978128496518
13.037938810051036 11.110119581658422 13.521140839642674 10.859018228147887
14.996457370329853 10.752272248615215 13.345508528717259 10.579005242826247
13.0287348030385 10.81529675167646 13.275829251377 10.460270761075149
13.1035438225232 11.192008258792368 13.195615697878253 10.868587496017359


In [184]:
mean_val = np.mean(rmse_err)
print(mean_val)
stdval = np.std(rmse_err)
print(stdval)

0.15983430204657595
0.04157063815376416
